# 00 · Inspección de CoVoST 2 (español → inglés)

**Objetivo:** inspeccionar el metadata oficial de [CoVoST 2](https://github.com/facebookresearch/covost) para la dirección `es → en` **sin descargar Common Voice**.

Qué hacemos y qué NO:

- ✅ Descargar únicamente el TSV de traducciones (`covost_v2.es_en.tsv`, ~3 MB).
- ✅ Cargar y explorar columnas, valores nulos, splits y ejemplos reproducibles.
- ✅ Guardar una muestra CSV reducida (100 registros) para inspección manual.
- ❌ No se descarga Common Voice (ni clips de audio ni `validated.tsv`).

## 1. Entorno y dependencias

In [6]:
# --- Entorno: raíz del proyecto y detección de Google Colab -----------------
import os
import sys
from pathlib import Path

def _is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

# Ruta raíz del repositorio. En Colab, si has montado Drive o subido la carpeta,
# ajusta PROJECT_ROOT a la ubicación real del proyecto.

PROJECT_ROOT_OVERRIDE: Path | None = None

def _is_project_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'requirements' / 'dataset.txt').is_file()

def _find_drive_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        return PROJECT_ROOT_OVERRIDE.expanduser().resolve()
    drive_root = Path('/content/drive')
    matches = sorted({src_dir.parent for src_dir in drive_root.glob('**/src')
                      if _is_project_root(src_dir.parent)})
    if len(matches) == 1:
        return matches[0]
    if matches:
        found = '\n - '.join(str(path) for path in matches)
        raise SystemExit(f'Se encontraron varios proyectos:\n - {found}\nAsigna uno a PROJECT_ROOT_OVERRIDE.')
    raise SystemExit('No se encontro IA-Proyecto. Monta la cuenta correcta de Drive o asigna PROJECT_ROOT_OVERRIDE.')

if _is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = _find_drive_project_root()
else:
    PROJECT_ROOT = Path.cwd()

if not _is_project_root(PROJECT_ROOT):
    raise SystemExit(f'No se encontro la raiz del proyecto en {PROJECT_ROOT}.')

if not (PROJECT_ROOT / "src").exists():
    raise SystemExit(
        f"No se encontró el repositorio en {PROJECT_ROOT}. "
        "Ajusta PROJECT_ROOT para que apunte a la carpeta IA-Proyecto."
    )

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)

Mounted at /content/drive


SystemExit: No se encontro IA-Proyecto. Monta la cuenta correcta de Drive o asigna PROJECT_ROOT_OVERRIDE.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
%pip install --quiet -r requirements/dataset.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements/dataset.txt'


### 1.1 Imports y configuración

In [ ]:
import pandas as pd
from IPython.display import display

from src.dataset import covost

SEED = covost.SEED
print("pandas", pd.__version__)

ModuleNotFoundError: No module named 'src'

## 2. Descarga del metadata oficial

La URL oficial del README de CoVoST 2 (`dl.fbaipublicfiles.com/covost/covost_v2.es_en.tsv.tar.gz`, ~3.2 MB) contiene un único TSV. La función **reutiliza el TSV** si ya se descargó, de modo que re-ejecutar el notebook no descarga de nuevo.

In [ ]:
tsv_path = covost.download_covost_metadata()
print("TSV listo en:", tsv_path)

NameError: name 'covost' is not defined

In [ ]:
df = covost.load_covost_metadata(tsv_path)
print("Dimensiones (filas, columnas):", df.shape)

NameError: name 'covost' is not defined

## 3. Inspección del metadata

### 3.1 Columnas disponibles

El TSV real de CoVoST 2 tiene **3 columnas**: `path`, `translation` y `split`.

In [ ]:
reports = covost.inspect_covost_metadata(df)
display(reports["columns"])

### 3.2 Valores nulos

In [ ]:
display(reports["missing_values"])

### 3.3 Distribución por split

Además de `train`/`dev`/`test`, el split actual incluye variantes `_covost` (muestras adicionales del paper) y `_dup` (duplicados). Para entrenamiento se suele agrupar por el nombre base de la partición.

In [ ]:
display(reports["split_distribution"])

### 3.4 Cantidad total de registros

In [ ]:
display(pd.DataFrame({"metric": ["total_registros"], "value": [len(df)]}))

### 3.5 Registros de ejemplo

In [ ]:
print("Primeros 5 registros:")
display(df.head(5))

In [ ]:
print("Ejemplos aleatorios reproducibles (seed=42):")
display(covost.random_examples(df, n=5))

In [ ]:
print("Ejemplos aleatorios reproducibles (seed=123):")
display(covost.random_examples(df, n=5, seed=123))

## 4. Columnas y su relación con Common Voice

CoVoST 2 se construye sobre grabaciones de Common Voice. El TSV de traducciones **no contiene** la transcripción en español (`sentence`) ni el identificador del hablante (`client_id`): ambas provienen del `validated.tsv` de Common Voice (versión 4). Para reconstruir el corpus paralelo completo hay que unir ambos metadatos **en texto**, sin necesidad de descargar audio.

In [ ]:
display(pd.DataFrame([
    {"columna": "path", "origen": "Common Voice (clip)", "uso": "ruta del archivo de audio"},
    {"columna": "translation", "origen": "CoVoST 2", "uso": "traducción al inglés"},
    {"columna": "split", "origen": "CoVoST 2", "uso": "train / dev / test (con variantes)"},
    {"columna": "sentence", "origen": "Common Voice (validated.tsv)", "uso": "transcripción en español — NO está en el TSV"},
    {"columna": "client_id", "origen": "Common Voice (validated.tsv)", "uso": "identificador del hablante — NO está en el TSV"},
]))

In [ ]:
display(pd.DataFrame({"split_value": sorted(df["split"].unique())}))

## 5. Muestra CSV reducida (inspección)

Se guardan 100 registros aleatorios en `data/external/translation/covost2_es_en_sample.csv` (UTF-8, semilla fija). Recuerda: el CSV solo tiene texto, no audio.

In [ ]:
sample = covost.sample_covost_metadata(df, n=100, seed=SEED)
print("Muestra creada:", len(sample), "registros")
display(sample.head())

## 6. Resumen y uso previsto

In [ ]:
summary = pd.DataFrame({
    "metrica": [
        "total_registros",
        "traducciones al inglés no nulas",
        "valores distintos en split",
        "audio descargado",
    ],
    "valor": [
        len(df),
        int(df["translation"].notna().sum()),
        df["split"].nunique(),
        "No (solo metadata)",
    ],
})
display(summary)

**¿Para qué sirve CoVoST 2 en este proyecto?**

- **A. Corpus de traducción es→en para fine-tuning del NMT (text→text):** uniendo este TSV con `validated.tsv` de Common Voice se reconstruyen los pares `(sentence_es, translation_en)` con ~112k registros. CoVoST 2 aporta la traducción y el split; Common Voice aporta la transcripción. Solo se usan metadatos de texto, no audio.
- **B. Dataset de evaluación Speech-to-Speech:** NO se usa CoVoST 2; para eso se construye una pequeña muestra con audio mediante FLEURS (notebooks 01–03).

Conclusión: CoVoST 2 es la futura fuente de entrenamiento del traductor, no parte del evaluador.